# E5 — Maximum stable coupling stiffness

`e5_kmax.py` appends every K step to `kmax.csv`; K_max of a ladder (method, delay, grip) is its last
stable K before the first unstable one (`down` search: the first stable step from the top). Expected:
flat for the model-based methods (set by the device and the 1 kHz loop), falling with T for M1.

Grips: `held` (the hand damps the loop), `released` (the first session: dropped and picked up 3 times),
`dropped` (let go of once, hands off: the worst case, added 2026-10-27). Held, M1 was stable at
K = 1000 even at 2.6 s: P-P under delay goes unstable slowly (~1/(2T), 0.2 Hz at 2.6 s), which the
hand damps and the HF RMS (> 8 Hz) cannot see. With the handle dropped, the slow instability shows as
the loop throwing the free handle off the table: unstable if it was thrown more than 10 mm above it
(`rebound_mm`), from 2 s after its first touch to the end.

In [ ]:
KMAX_CSV = "~/data/franka/rim-delays/robot/e5/kmax.csv"
SHOW_K = 1000  # the time series: dropped runs at this K (the frozen coupling)

In [ ]:
import os

os.environ.setdefault("JAX_PLATFORMS", "cpu")  # before anything imports JAX: the GPU stays free for fr3_teleop's IK
import json
import sys
from pathlib import Path

HERE = Path.cwd()
sys.path[:0] = [str(HERE), str(HERE.parent)]
import matplotlib.pyplot as plt  # noqa: I001
import numpy as np
import pandas as pd

import metrics as M
import plots as P
from runs import load_run, surface_of

%matplotlib inline
%load_ext autoreload
# pick up edits to exp_metrics / plots / common without a kernel restart
%autoreload 2
pd.set_option("display.precision", 3)

steps = pd.read_csv(Path(KMAX_CSV).expanduser())
steps["T"] = steps.rtt.map(P.delay_key)
# a step run again (e.g. the dropped M5 run at 8 Hz, superseded by the 50 Hz filter) keeps its latest run
steps = steps.sort_values("run").drop_duplicates(["method", "rtt", "grip", "K"], keep="last").reset_index(drop=True)
# the dropped-handle metrics live in each run's metrics.json (kmax.csv keeps its first columns)
extra = ["rebounds", "rebound_mm", "depth_max_mm"]
for c in extra:
    steps[c] = [json.loads((Path(r) / "analysis" / "metrics.json").read_text()).get(c, np.nan) for r in steps.run]
steps[["method", "T", "grip", "K", "search", "hf_rms_N", "guard_trips", "E_min_J", *extra, "stable"]]

In [ ]:
def k_max(g):
    g = g.sort_values("K")
    unstable = g[~g.stable.astype(bool)]
    ok = g[g.K < unstable.K.min()] if len(unstable) else g
    if g.search.iloc[0] == "down":  # from the top: K_max is the stable step, every step above it was unstable
        ok = g[g.stable.astype(bool)]
    return ok.K.max() if len(ok) else float("nan")


kmax = steps.groupby(["method", "T", "grip"]).apply(k_max, include_groups=False).rename("K_max").reset_index()
kmax = kmax.merge(steps.groupby(["method", "T", "grip"]).K.min().rename("K_min_tested").reset_index())
kmax  # NaN: no stable step in the ladder (K_max below the lowest K tested)

In [ ]:
GRIPS = {"held": "-", "released": ":", "dropped": "--"}
fig, ax = plt.subplots(figsize=(7, 4))
xs = P.delay_axis(ax, kmax["T"])
for m in P.methods_in(kmax):
    for grip, ls in GRIPS.items():
        d = kmax[(kmax.method == m) & (kmax.grip == grip)].set_index("T").reindex([k for k in xs]).dropna(how="all")
        if d.empty:
            continue
        x = [xs[k] for k in d.index]
        ax.plot(x, d.K_max, "o" + ls, color=P.color(m), label=f"{P.LABELS.get(m, m)}, {grip}")
        none = d.K_max.isna()  # no stable step: drawn below the lowest K tested
        ax.plot(np.array(x)[none], 0.8 * d.K_min_tested[none], "v", ms=9, color=P.color(m), mfc="white")
ax.set_ylabel("K_max [N/m]  (▽: below the lowest K tested)")
ax.set_ylim(bottom=0)
ax.legend(fontsize=8)
fig.tight_layout()

## The dropped handle, M1 vs M5

At `SHOW_K` and each delay with a dropped run: the free handle's height above the table (top) and the
felt force (bottom), from its first touch. M5's wall is local, so the handle rests on it; M1's wall is the
delayed robot, which arrives a round trip late and keeps pushing a round trip after the handle has left —
the loop throws the handle. Shaded: the first 2 s (the hand still on it).

In [ ]:
drop = steps[(steps.grip == "dropped") & (steps.K == SHOW_K)].sort_values(["T", "method"])
if drop.empty:
    print(f"no dropped runs at K = {SHOW_K}")
else:
    delays = [d for d in P.DELAYS if d in set(drop["T"])]
    fig, axes = plt.subplots(2, len(delays), figsize=(6 * len(delays), 5.5), sharex=True, sharey="row", squeeze=False)
    for j, T in enumerate(delays):
        for _, row in drop[drop["T"] == T].iterrows():
            run = load_run(row.run)
            hs = M.haptic_side(run)
            z, s = hs["x_l"][:, 2], surface_of(run["meta"])
            below = np.flatnonzero(z < s)
            if not len(below):
                continue
            t = hs["t"] - hs["t"][below[0]]
            lab = f"{P.LABELS.get(row.method, row.method)}  ({'stable' if row.stable else 'UNSTABLE'})"
            axes[0, j].plot(t, 1e3 * (z - s), lw=1.5, color=P.color(row.method), label=lab)
            axes[1, j].plot(t, hs["f"][:, 2], lw=1.5, color=P.color(row.method))
        for ax in axes[:, j]:
            ax.axvspan(-1, 2.0, color="0.92", zorder=0)
            ax.axhline(0, color="0.6", lw=1, zorder=0)
        axes[0, j].axhline(10, color="0.6", lw=1, ls=":", zorder=0)  # the rebound limit
        axes[0, j].set_title(f"T = {T if T == '5g' else T + ' ms'}, K = {SHOW_K} N/m, handle dropped", loc="left", fontsize=10)
        axes[0, j].legend(fontsize=8, frameon=False)
        axes[1, j].set_xlabel("time from the first touch [s]")
        axes[1, j].set_xlim(-1, None)
    axes[0, 0].set_ylabel("handle height above the table [mm]")
    axes[1, 0].set_ylabel("felt force, z [N]")
    fig.tight_layout()